# LangGraph Workflow with Semantra Core

This notebook demonstrates how to wire the `semantra-core` service
protocols into a LangGraph state machine.

It uses the **in-memory reference implementations** by default. You can
swap in the `BackendMappingEngine`, `BackendLLMService`, and
`BackendKnowledgeBase` from `semantra-backend-adapter` to run the same
graph against the real Semantra backend.

In [ ]:
# Install the SDK with the LangGraph extra (run once)
# !pip install -e "../semantra-core[langgraph]"

In [ ]:
from semantra_core.models.schema import (
    DatasetHandle,
    SchemaProfile,
    ColumnProfile,
)
from semantra_core.services import (
    InMemoryMappingEngine,
    BoundedLLMService,
)
from semantra_core.langgraph_workflow import build_semantra_graph

## 1. Build a minimal source and target

In [ ]:
source = DatasetHandle(
    dataset_id="src",
    dataset_name="customer_src",
    schema_profile=SchemaProfile(
        dataset_id="src",
        dataset_name="customer_src",
        row_count=10,
        columns=[
            ColumnProfile(
                name="cust_id",
                normalized_name="cust_id",
                dtype="str",
                null_ratio=0.0,
                unique_ratio=1.0,
                non_null_count=10,
            )
        ],
    ),
)
target = SchemaProfile(
    dataset_id="tgt",
    dataset_name="customer_tgt",
    row_count=0,
    columns=[
        ColumnProfile(
            name="customer_key",
            normalized_name="customer_key",
            dtype="str",
            null_ratio=0.0,
            unique_ratio=0.0,
            non_null_count=0,
        )
    ],
)
print("Source and target ready.")

## 2. Build the LangGraph state machine

In [ ]:
engine = InMemoryMappingEngine()
llm = BoundedLLMService()

graph = build_semantra_graph(engine=engine, llm=llm)
print("Graph compiled. Nodes:", list(graph.nodes.keys()) if hasattr(graph, "nodes") else graph)

## 3. Run the graph

In [ ]:
result = graph.invoke({"source": source, "target": target})
print("Result:", result)

## 4. Swap in the backend adapters (optional)

If you have the `semantra-backend-adapter` installed and the real backend
is available, you can replace the in-memory implementations with the
backend-backed ones:

In [ ]:
try:
    from semantra_backend_adapter import create_backend_adapters
    adapters = create_backend_adapters()
    graph_real = build_semantra_graph(
        engine=adapters["engine"],
        llm=adapters["llm"],
        knowledge=adapters["knowledge"],
    )
    result_real = graph_real.invoke({"source": source, "target": target})
    print("Backend-backed graph result:", result_real)
except Exception as e:
    print(f"Could not run backend-backed graph: {e}")

## 5. Use async backend mapping with LangChain tools

If you want the backend to drive non-blocking mapping inside an async agent, request the async mapping engine and pass it to `build_semantra_tools`.

In [ ]:
try:
    from semantra_agent.langchain_tools import build_semantra_tools

    adapters = create_backend_adapters(include_async_engine=True)
    tools = build_semantra_tools(
        async_engine=adapters["async_engine"],
        llm=adapters["llm"],
    )
    print("Built async backend LangChain tools:", [tool.name for tool in tools])
except Exception as e:
    print(f"Could not build async backend LangChain tools: {e}")

## What this notebook shows

This notebook demonstrates a hybrid Semantra workflow:

1. LangGraph state machine execution with in-memory services.
2. Swap-in of the real backend adapters.
3. Async backend mapping exposed as `async_engine`.
